# Noisy Channel Image Rescue: train the model (free Colab / Kaggle)

This notebook trains the small restoration CNN and gives you the two files the app needs:
`models/model.pt` and `models/model.onnx`.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells from top to bottom.
Total time is roughly 15-20 minutes (dataset download + training).

## 1. Get the code
Choose **one** option and run only that cell.

In [ ]:
# OPTION A: clone from your GitHub (after you pushed the repo). Replace YOUR_USERNAME.
!git clone https://github.com/YOUR_USERNAME/image-restoration-telecom.git
%cd image-restoration-telecom

In [ ]:
# OPTION B: no GitHub yet? Upload image-restoration-telecom.zip (the file I gave you).
from google.colab import files
up = files.upload()
!unzip -q -o image-restoration-telecom.zip
%cd image-restoration-telecom

## 2. Install and check

In [ ]:
!pip install -q onnx onnxscript onnxruntime opencv-python-headless scikit-image pytest
import torch
print("PyTorch", torch.__version__, "| GPU available:", torch.cuda.is_available())

In [ ]:
# quick self-test of the channel simulator, classic filters and the model code
!python -m pytest -q tests

## 3. Smoke test (about 1 minute)
Catches problems early, before the long run. Uses CIFAR-10 (small download) and 300 images.

In [ ]:
!python train.py --dataset cifar10 --epochs 1 --max-images 300 --out models_smoke --workers 2

## 4. Real training: the final model (v2), about 18 minutes on a T4
This is the exact recipe of the final model: CIFAR-10 (20,000 images), 40 epochs, learning rate **5e-4**.

Why not the defaults? STL-10 is a 2.6 GB download that was far too slow on free Colab. The first
CIFAR-10 run used a learning rate of 2e-3 and stalled: it learned to fill lost blocks but not to remove
noise (validation 24.7 dB). Lowering the learning rate to 5e-4 fixed it (validation 30.5 dB).
`model.pt` is saved whenever validation quality improves.

In [ ]:
!python train.py --dataset cifar10 --max-images 20000 --repeat 2 --epochs 40 --lr 5e-4 --out models --workers 2

## 5. Evaluate: damaged vs classic OpenCV vs AI
Creates the tables and graphs for your report.

In [ ]:
!python evaluate.py --out results

In [ ]:
from IPython.display import Image, Markdown, display
display(Image("results/curves.png"))
display(Image("results/examples.png"))
print(open("models/train_summary.json").read())

In [ ]:
display(Markdown(open("results/summary.md").read()))

## 5b. (Recommended) Test on unseen photos, with statistics
Zip 15-30 photos that were NOT used for training, directly (no subfolder) as `my_photos.zip`, upload it, then run the cells below.
The report prints a 95% confidence interval and the number of photos where the AI wins, so the claim does not
rest on a handful of images.

In [ ]:
from google.colab import files
files.upload()   # choose my_photos.zip

In [ ]:
!unzip -q -j -o my_photos.zip -d my_photos
!python evaluate.py --model models/model.onnx --images my_photos --seeds 3 --out results_photos

## 6. Download your files
Download the zip, then copy `models/model.onnx` (and `model.pt`) into the `models/` folder of your repo
and `results/` into the repo's `results/` folder. Commit and push. The app picks the model up automatically.

In [ ]:
!zip -q -r trained_outputs.zip models results
from google.colab import files
files.download("trained_outputs.zip")

## 7. Optional experiment (negative result): enlarge the training images 2x
Hypothesis: CIFAR images are tiny and very detailed per pixel, unlike normal photos, so enlarging them 2x
might help the model generalise. **It did not.** On 28 unseen photos this model (v3) was better than v2
only on salt-and-pepper noise, and worse on Gaussian noise, packet loss, JPEG and combined damage, so v2 was
kept. Run it only if you want to reproduce this ablation.

In [ ]:
import numpy as np, cv2, os
from torchvision.datasets import CIFAR10
data = CIFAR10(root="data", train=True, download=True).data
os.makedirs("cifar_up", exist_ok=True)
for i in range(20000):
    big = cv2.resize(data[i], (64, 64), interpolation=cv2.INTER_CUBIC)
    cv2.imwrite(f"cifar_up/{i:05d}.png", cv2.cvtColor(big, cv2.COLOR_RGB2BGR))
print("wrote", len(os.listdir("cifar_up")), "images")

In [ ]:
!python train.py --dataset folder --data-dir cifar_up --patch 48 --repeat 2 --epochs 40 --lr 5e-4 --out models_v3 --workers 2
!python evaluate.py --model models_v3/model.onnx --images my_photos --seeds 3 --out results_photos_v3